<a href="https://colab.research.google.com/github/davidlealo/sic_ia_2026/blob/main/clases/sic_2026_ia_Clase_21.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Revisión de grupos



# Datos sintéticos para clasificar cantos de aves chilenas

Material de apoyo para estudiantes. Resume una conversación de trabajo sobre el modelo EfficientNet-B1-BirdSet-XCL, la generación de datos sintéticos de cantos de aves y el diseño de un experimento para evaluar si el audio sintético mejora la clasificación de especies chilenas con pocas grabaciones.

Fecha: noviembre de 2026.

---

## 1. Qué es EfficientNet-B1-BirdSet-XCL

Es un clasificador de cantos de aves publicado por el DBD-research-group como parte del benchmark **BirdSet** (Rauch et al., ICLR 2025).

- **Arquitectura:** adapta EfficientNet-B1 para clasificar segmentos de audio de 5 segundos en 9.736 especies y produce embeddings de 1.280 dimensiones. En Hugging Face aparece etiquetado como "Image Classification" porque opera sobre espectrogramas, que trata como imágenes.
- **Datos de entrenamiento (XCL):** Xeno-Canto Large, con alrededor de 530 mil grabaciones de unas 10.000 especies. Los pesos se inicializan desde ImageNet y luego se entrenan de forma supervisada sobre XCL. Existe también una variante **XCM**, entrenada con una versión mediana del dataset.
- **Contexto del benchmark:** BirdSet entrena con grabaciones focales (una especie dominante por grabación) y evalúa en paisajes sonoros reales, donde la tarea es multi-etiqueta. El salto entre ambos dominios es lo que el benchmark busca medir.
- **Familia de modelos:** el mismo grupo publica versiones XCL y XCM de Wav2Vec2-Base, EfficientNet-B1, AST y ConvNeXT-Base. EfficientNet-B1 es la opción más liviana (del orden de 7,8 millones de parámetros en la arquitectura base), útil para inferencia en hardware modesto o como extractor de embeddings.
- **Advertencia práctica:** el modelo espera audio preprocesado según BirdSet (32 kHz y espectrograma mel con parámetros específicos). Conviene usar la librería `birdset` o replicar exactamente ese preprocesamiento; de lo contrario, el rendimiento cae bastante.

---

## 2. Puede este modelo generar sonidos

No. Es un modelo discriminativo: recibe un espectrograma y devuelve probabilidades por especie. No tiene decodificador que produzca audio. Sin embargo, cumple roles útiles en un pipeline de datos sintéticos:

1. **Filtro de calidad:** se clasifica cada muestra sintética y se descartan las que no se reconocen como la especie objetivo o tienen baja confianza. Solo funciona si la especie está entre sus 9.736 clases; algunas aves chilenas raras o endémicas pueden faltar o tener pocas grabaciones en Xeno-Canto.
2. **Métrica de fidelidad:** sus embeddings permiten comparar la distribución de audio real con la de audio sintético, al estilo de la Fréchet Audio Distance, pero en un espacio de representación específico de aves.
3. **Backbone del clasificador final:** lo más probable es que el experimento termine afinando un modelo como este, no entrenando uno desde cero.

---

## 3. Estrategias para aumentar datos

De menor a mayor complejidad:

1. **Aumentación clásica:** cambio de tono leve, estiramiento temporal, ganancia, SpecAugment y, sobre todo, mezcla de vocalizaciones con ruido de fondo real de hábitats chilenos (viento, lluvia, otras especies, insectos). Suele ser lo que más rinde, porque el problema central es el salto entre grabaciones focales limpias y paisajes sonoros ruidosos.
2. **Recombinación de sílabas:** segmentar cantos en unidades y armar secuencias nuevas. Funciona en especies con repertorio estructurado, aunque puede producir secuencias biológicamente inverosímiles.
3. **Modelos generativos:** GANs sobre espectrogramas o modelos de difusión condicionados, idealmente partiendo de un modelo preentrenado y afinándolo con las grabaciones disponibles.

**Riesgo principal de los modelos generativos:** con muy pocas grabaciones por especie, tienden a memorizar. Producen variaciones cosméticas de las mismas grabaciones y no diversidad real, lo que puede inflar artificialmente los resultados.

---

## 4. Diseño experimental recomendado

- **Separar por grabación, no por segmento.** Si se generan sintéticos a partir de una grabación, esa grabación y todo lo derivado de ella no pueden aparecer en el conjunto de prueba. Es el error más común en este tipo de estudios.
- **Evaluar solo con audio real**, idealmente paisajes sonoros chilenos y no grabaciones focales.
- **Comparar contra baselines fuertes:**
  - solo datos reales;
  - datos reales con aumentación clásica;
  - datos reales con sintéticos;
  - clasificación few-shot con embeddings congelados (prototipos por especie), que con pocos datos suele ser difícil de superar.
- **Barrer la proporción real/sintético** (por ejemplo 1:0, 1:1, 1:5, 1:10) para identificar dónde la ganancia se estabiliza o empieza a degradar.
- **Medir por especie**, no solo el promedio.

**Criterio de interpretación:** si los sintéticos superan a "solo reales" pero no a "aumentación clásica", la conclusión honesta es que el aporte proviene de la regularización y no de la síntesis en sí.

---

## 5. Modelos generativos abiertos recomendados

### 5.1 ECOGEN (recomendado para empezar)

Desarrollado en la Université de Moncton y publicado en *Methods in Ecology and Evolution*. Su objetivo explícito es aumentar las muestras de clases de cantos subrepresentadas. Genera espectrogramas con técnicas de generación de imágenes y luego los convierte de vuelta a audio. Los autores reportan que añadir muestras de ECOGEN mejoró la precisión de un clasificador en 12% en promedio y superó a la aumentación clásica en el 80% de los casos. Es open source y funciona en hardware básico.

Observación: ese 80% implica que en uno de cada cinco casos la aumentación clásica fue igual o mejor, por lo que el baseline clásico sigue siendo imprescindible.

### 5.2 BirdDiff (más controlable, más exigente)

Basado en DiffWave, un modelo de difusión que genera directamente la forma de onda. Se condiciona con coeficientes MFCC, etiquetas de especie y descripciones textuales, y está pensado para grabaciones de campo ruidosas. El código y el dataset están disponibles en GitHub.

Advertencia de los propios autores: sin una etapa previa de mejora de la relación señal/ruido, el modelo no genera cantos inteligibles ni específicos de la especie.

### 5.3 Difusión sobre espectrogramas con Stable Diffusion

Un trabajo de 2024 mostró que aumentar la clasificación de especies con espectrogramas sintéticos generados con Stable Diffusion mejora el rendimiento en ambientes ruidosos frente a usar solo los datos originales. Es una buena opción si ya se maneja ese ecosistema y se quiere afinar con LoRA.

### 5.4 Alternativa general

Stable Audio Open es un modelo de texto a audio con pesos públicos que puede afinarse, pero no está entrenado específicamente en aves y su licencia comunitaria tiene condiciones de uso que deben revisarse antes de publicar.

---

## 6. Ruta sugerida

1. Baseline con aumentación clásica y mezcla con ruido de fondo chileno.
2. ECOGEN como primer generador, por bajo costo y evidencia previa.
3. BirdDiff como generador fuerte, si los resultados de ECOGEN lo justifican.
4. Evaluación con EfficientNet-B1-BirdSet-XCL (o ConvNeXt de BirdSet) afinado, probando siempre solo con grabaciones reales separadas por grabación.

Con muy pocas grabaciones por especie (por ejemplo, menos de 10 a 20), conviene preentrenar el generador con muchas especies, por ejemplo aves sudamericanas de Xeno-Canto, y luego condicionar o afinar para las especies chilenas escasas, en lugar de entrenar un generador por especie.

---

## Referencias y enlaces

### Modelo y benchmark

- Modelo en Hugging Face: [DBD-research-group/EfficientNet-B1-BirdSet-XCL](https://huggingface.co/DBD-research-group/EfficientNet-B1-BirdSet-XCL)
- Colección de modelos BirdSet: [DBD-research-group en Hugging Face](https://huggingface.co/DBD-research-group)
- Rauch, L. et al. BirdSet: A Large-Scale Dataset for Audio Classification in Avian Bioacoustics. ICLR 2025. [arXiv:2403.10380](https://arxiv.org/abs/2403.10380)
- Presentación de BirdSet en ICLR 2025: [diapositivas (PDF)](https://iclr.cc/media/iclr-2025/Slides/28986.pdf)
- Resumen técnico del modelo: [BirdSetEfficientNetB1 en Emergent Mind](https://www.emergentmind.com/topics/birdsetefficientnetb1)
- Uncertainty Calibration of Multi-Label Bird Sound Classifiers (compara modelos BirdSet y Perch v2): [arXiv:2511.08261](https://arxiv.org/abs/2511.08261)

### Generación de cantos de aves

- Guei, A.-C., Christin, S., Lecomte, N. y Hervet, É. (2024). ECOGEN: Bird sounds generation using deep learning. *Methods in Ecology and Evolution*, 15(1), 69-79. [doi:10.1111/2041-210X.14239](https://doi.org/10.1111/2041-210X.14239)
- Nota de prensa sobre ECOGEN: [NSERC](https://nserc-crsng.canada.ca/en/new-deep-learning-ai-tool-helps-ecologists-monitor-rare-birds-through-their-songs)
- Song, T. et al. Towards High-Fidelity and Controllable Bioacoustic Generation via Enhanced Diffusion Learning (BirdDiff). [arXiv:2509.00318](https://arxiv.org/abs/2509.00318) y [versión en ScienceDirect](https://www.sciencedirect.com/science/article/pii/S1574954126003249)
- Código y datos de BirdDiff: [GitHub](https://github.com/BirdDiff/Bioacoustics-Generation/releases/tag/V1)
- Generative AI-based data augmentation for improved bioacoustic classification in noisy environments: [arXiv:2412.01530](https://arxiv.org/abs/2412.01530)

### Otros recursos

- Xeno-Canto, repositorio colaborativo de grabaciones de aves: [xeno-canto.org](https://xeno-canto.org)
- Kong, Z. et al. DiffWave: A Versatile Diffusion Model for Audio Synthesis. ICLR 2021. [arXiv:2009.09761](https://arxiv.org/abs/2009.09761)

# Selección de especies chilenas para probar datos sintéticos

Material de apoyo para estudiantes. Continúa el documento "Datos sintéticos para clasificar cantos de aves chilenas" y resume cómo elegir una especie chilena para el experimento, un caso documentado en la literatura, lo que se sabe sobre umbrales de datos y un script para obtener métricas desde Xeno-Canto.

Fecha: noviembre de 2026.

---

## 1. Criterios para elegir la especie

La especie ideal para el experimento cumple tres condiciones:

1. **El clasificador no la reconoce bien**, o hay razones para esperar que la confunda.
2. **Tiene datos moderados, no escasos.** Esto permite reducir artificialmente el set de entrenamiento (por ejemplo a 3, 5, 10 y 20 grabaciones) y completarlo con sintéticos, manteniendo fijo un set de prueba real.
3. **Existen grabaciones que el modelo no vio**, para evaluar sin filtración.

No existe, hasta donde se pudo revisar, una tabla publicada del rendimiento de EfficientNet-B1-BirdSet-XCL por especie chilena. Los sets de prueba de BirdSet son de otras regiones. Por eso, el desempeño sobre una especie chilena debe medirse empíricamente.

---

## 2. El problema de la filtración

El modelo se entrenó con XCL, que proviene íntegramente de Xeno-Canto. Si se evalúa con grabaciones de Xeno-Canto anteriores al corte de BirdSet, es probable que el modelo ya las haya escuchado y el rendimiento se vea artificialmente bueno.

Hay dos salidas:

1. **Evaluar con audio no visto:** grabaciones subidas a Xeno-Canto después del corte del dataset (revisar la fecha en el paper de BirdSet), grabaciones de Macaulay Library o grabaciones propias de terreno.
2. **Usar una especie fuera de las 9.736 clases del modelo.** Así se trabaja con una clase realmente nueva. La lista de clases está en el archivo `config.json` del repositorio del modelo en Hugging Face (campo `id2label`).

---

## 3. Caso documentado: el chuncho (*Glaucidium nana*) en Valdivia

**Fuente:** Morales, G., Vargas, V., Espejo, D., Poblete, V., Tomasevic, J. A., Otondo, F. y Navedo, J. G. (2022). Method for passive acoustic monitoring of bird communities using UMAP and a deep neural network. *Ecological Informatics*, 72, 101909.

Es un trabajo de la Universidad Austral de Chile. Los autores entrenaron un clasificador para 16 especies con grabaciones pasivas de tres humedales urbanos de Valdivia. El modelo obtuvo un mAP de 0,97, con un AP mínimo de 0,79.

**Por qué es relevante para este experimento:**

- **Fue la especie con peor desempeño.** Las dos clases del chuncho fueron las curvas de precisión y recall más alejadas del ideal, y eran justamente las clases aumentadas con muestras simuladas.
- **Probaron algo muy parecido a lo que se quiere probar aquí.** Generaron variantes de cada muestra con reducción de energía, estiramiento temporal, cambio de tono y combinaciones, y las mezclaron con paisajes sonoros locales sin la especie.
- **Los autores proponen causas explícitas del fracaso:** inconsistencia entre muestras reales y simuladas, poca diferenciación entre clases de una misma especie y una ventana de dos segundos insuficiente para una vocalización que puede durar cuatro. BirdSet usa ventanas de 5 segundos, por lo que EfficientNet-B1-BirdSet-XCL ya corrige esa última limitación.
- **Tiene datos moderados.** Xeno-Canto registra 69 grabaciones en primer plano y 4 de fondo, con unos 53 minutos en total. Macaulay Library tiene además grabaciones recientes de Chile, posteriores a BirdSet, útiles para un set de prueba no visto.

**Pregunta de investigación que se desprende:** si la aumentación falló por la simulación en sí o por el diseño (ventana, clases, filtración), y si un generador más sofisticado o un modelo preentrenado grande resuelve el problema.

**Segundo caso del mismo paper:** el pidencito (*Porzana spiloptera*), especie vulnerable. Los autores simularon su llamada a partir de 34 grabaciones de Xeno-Canto registradas en Argentina y obtuvieron 1.224 muestras simuladas, pero reconocen que no pudieron evaluar bien esa simulación porque entrenamiento, validación y prueba quedaron vinculados. Es relevante conceptualmente, pero tiene muy pocos datos para este experimento.

**Salvedad:** el caso está documentado con un ResNet50 entrenado por los autores, no con EfficientNet-B1-BirdSet-XCL. El primer paso del experimento debería ser medir el modelo de BirdSet sobre el chuncho con audio no visto. Si funciona bien, eso ya es un resultado: un modelo grande preentrenado resolvería lo que la aumentación local no pudo.

**Advertencia de los propios autores:** el alto rendimiento probablemente está sesgado porque vocalizaciones de los mismos individuos quedaron en entrenamiento y prueba. Recomiendan reservar zonas de monitoreo distintas para validación y prueba.

---

## 4. Candidato alternativo: churrín del norte (*Scytalopus fuscus*)

- Es endémico de Chile central y habita quebradas con vegetación densa, por lo que se puede grabar cerca de Santiago para el set de prueba.
- Antes se consideraba subespecie del churrín de Magallanes (*Scytalopus magellanicus*) y se separó por diferencias vocales. El género *Scytalopus* es conocido por especies que suenan muy parecido, lo que hace esperable que un clasificador las confunda.
- Xeno-Canto registraba 43 grabaciones en primer plano y 6 de fondo, con unos 40 minutos en total (cifra a confirmar en el sitio).

Otra opción no verificada es la tenca (*Mimus thenca*): al ser imitadora, su repertorio se superpone con el de otras especies.

---

## 5. Sobre el umbral de "20 grabaciones de 2,5 minutos"

Este criterio circuló en el grupo, pero **no se pudo confirmar** en la literatura revisada.

- **BirdSet no fija un mínimo de grabaciones por especie.** Su protocolo limita a un máximo de 500 grabaciones por clase, sin mínimo, para incluir todas las especies de XCL. Es decir, especies con muy pocas grabaciones también entraron al entrenamiento.
- **Posible origen del número 20:** el artículo de los fundadores de Xeno-Canto (Vellinga y Planqué, 2015) indica que el número esperado de grabaciones por especie en la colección rondaba las 20. Es una estadística de la colección, no un umbral de desempeño.
- **La intuición tiene respaldo general, con matices.** En otro estudio, el desempeño por especie creció en general con el número de muestras, y las especies sin aciertos tenían entre 1 y 7 ejemplos. Pero algunas especies rindieron bien con pocos datos, por lo que el número de grabaciones no basta para predecir dónde falla un modelo.
- **BirdSet excluyó grabaciones con licencia "no derivadas" (ND).** Por eso, el número de grabaciones que el modelo realmente vio de una especie puede ser menor que el que muestra Xeno-Canto.

---

## 6. Métricas disponibles de especies chilenas

| Especie | Grabaciones en primer plano | Duración total |
|---|---|---|
| Churrín del norte (*Scytalopus fuscus*) | 43 | 40:17 |
| Chuncho (*Glaucidium nana*) | 69 | 52:46 |
| Cachudito (*Anairetes parulus*) | 97 | 39:50 |

Con el criterio tentativo (20 grabaciones y 50 minutos en total), las tres superan el número de grabaciones pero quedan bajo los 50 minutos. La parte de duración pesa más de lo que parece, porque muchas grabaciones de Xeno-Canto duran menos de un minuto.

Estas cifras provienen de las páginas de especie de Xeno-Canto y pueden haber cambiado. Deben verificarse en el sitio o con el script de la sección siguiente.

---

## 7. Script para obtener las métricas completas

El archivo `metricas_xc_chile.py` (incluido en el anexo) consulta la API v3 de Xeno-Canto, lista todas las especies con grabaciones en Chile y calcula por especie:

- total de grabaciones en Xeno-Canto y número registrado en Chile;
- minutos totales y mediana de duración;
- grabaciones de calidad A o B;
- número de grabadores distintos;
- estimación de cuántas grabaciones vio el modelo (sin licencia ND, anteriores al corte, con tope de 500);
- número de grabaciones posteriores al corte, candidatas a set de prueba;
- una marca `bajo_umbral` según los criterios configurables.

**Antes de ejecutarlo:**

1. Obtener una clave gratuita de la API v3 con una cuenta en xeno-canto.org y guardarla en la variable de entorno `XC_API_KEY`.
2. Ajustar `CORTE_BIRDSET` a la fecha del snapshot de Xeno-Canto que reporta el paper de BirdSet. El valor incluido es provisional.
3. Revisar la documentación vigente de la API por si cambió la sintaxis de los filtros.

**Criterio final:** tomar las especies marcadas como `bajo_umbral` que tengan suficientes grabaciones posteriores al corte, y medir el modelo sobre ellas. La elección sigue siendo empírica.

---

## 8. Cómo armar los sets

Con unas 45 a 70 grabaciones por especie:

- **Partición 60/20/20** (entrenamiento, validación, prueba).
- **Agrupar por grabación y por grabador.** Si un mismo grabador aporta muchas grabaciones y estas quedan repartidas entre entrenamiento y prueba, el modelo puede aprender su micrófono o su lugar de grabación en vez del ave.
- **Simular la escasez en entrenamiento** con 3, 5, 10 y 20 grabaciones, completando con sintéticos y manteniendo fijo el set de prueba real.
- **Repetir con varias particiones** (validación cruzada agrupada) y reportar media y varianza, porque con tan pocas grabaciones un único corte puede dar resultados engañosos.

---

## Referencias y enlaces

### Caso documentado en Chile

- Morales, G. et al. (2022). Method for passive acoustic monitoring of bird communities using UMAP and a deep neural network. *Ecological Informatics*, 72, 101909. [doi:10.1016/j.ecoinf.2022.101909](https://doi.org/10.1016/j.ecoinf.2022.101909). [PDF abierto en SoundLapse](https://soundlapse.net/wp-content/uploads/2023/11/Birds_communities_UMAP_deep_neural_networks-2022.pdf)

### BirdSet y Xeno-Canto

- Rauch, L. et al. BirdSet: A Large-Scale Dataset for Audio Classification in Avian Bioacoustics. ICLR 2025. [arXiv:2403.10380](https://arxiv.org/abs/2403.10380)
- Heinrich, R. et al. AudioProtoPNet: An interpretable deep learning model for bird sound classification (describe el protocolo de entrenamiento de BirdSet, incluido el tope de 500 grabaciones y la exclusión de licencias ND). [arXiv:2404.10420](https://arxiv.org/abs/2404.10420)
- Vellinga, W.-P. y Planqué, R. (2015). The Xeno-canto collection and its relation to sound recognition and classification. CLEF Working Notes. [PDF](https://ceur-ws.org/Vol-1391/166-CR.pdf)
- Relación entre datos de entrenamiento y desempeño por especie: Localized time-frequency representation learning for bioacoustic classification in complex soundscapes. [arXiv:2502.13440](https://arxiv.org/abs/2502.13440)
- Modelo en Hugging Face: [DBD-research-group/EfficientNet-B1-BirdSet-XCL](https://huggingface.co/DBD-research-group/EfficientNet-B1-BirdSet-XCL)
- Documentación de la API de Xeno-Canto: [xeno-canto.org/explore/api](https://xeno-canto.org/explore/api)

### Especies

- Xeno-Canto, chuncho: [Glaucidium nana](https://www.xeno-canto.org/species/Glaucidium-nana)
- Xeno-Canto, churrín del norte: [Scytalopus fuscus](https://www.xeno-canto.org/species/scytalopus-fuscus)
- Wikipedia, churrín del norte: [Dusky tapaculo](https://en.wikipedia.org/wiki/Dusky_tapaculo)
- Macaulay Library, chuncho en El Quisco (2025): [ML642738018](https://macaulaylibrary.org/asset/642738018)
- Macaulay Library, chuncho en Puerto Varas (2025): [ML636369417](https://macaulaylibrary.org/asset/636369417)

---

## Anexo: metricas_xc_chile.py

```python
"""
Métricas por especie de aves registradas en Chile en Xeno-Canto,
pensadas para elegir especies "bajo umbral" para EfficientNet-B1-BirdSet-XCL.

Uso:
    pip install requests
    export XC_API_KEY="tu_clave"      # clave gratuita de tu cuenta en xeno-canto.org
    python metricas_xc_chile.py

Salida: metricas_xc_chile.csv

Notas importantes (verificar antes de usar los resultados):
- La API v3 de Xeno-Canto requiere clave. Revisen la documentación vigente
  (https://xeno-canto.org/explore/api) por si cambió la sintaxis de los tags.
- BirdSet excluyó las grabaciones con licencia "no derivadas" (ND) y limitó
  a 500 grabaciones por especie. Por eso n_xcl_aprox cuenta solo grabaciones
  sin ND subidas antes de CORTE_BIRDSET. Es una aproximación: la lista exacta
  está en el dataset XCL de BirdSet en Hugging Face.
- CORTE_BIRDSET es un supuesto: ajústenlo a la fecha del snapshot de XC
  que reporta el paper de BirdSet.
"""

import csv
import os
import statistics
import time

import requests

API = "https://xeno-canto.org/api/3/recordings"
KEY = os.environ["XC_API_KEY"]

CORTE_BIRDSET = "2024-01-01"   # AJUSTAR: fecha del snapshot de XC usado por BirdSet
UMBRAL_GRABACIONES = 20        # criterio a confirmar
UMBRAL_MINUTOS = 50            # 20 grabaciones x 2,5 min, criterio a confirmar
MAX_XCL = 500                  # tope por especie usado en BirdSet


def consultar(query):
    """Descarga todas las páginas de resultados para una consulta."""
    pagina, grabaciones = 1, []
    while True:
        r = requests.get(
            API,
            params={"query": query, "key": KEY, "page": pagina, "per_page": 500},
            timeout=60,
        )
        r.raise_for_status()
        datos = r.json()
        grabaciones.extend(datos.get("recordings", []))
        if pagina >= int(datos.get("numPages", 1)):
            return grabaciones
        pagina += 1
        time.sleep(1)  # respetar la API


def segundos(duracion):
    """Convierte 'm:ss' o 'h:mm:ss' a segundos."""
    total = 0
    for parte in duracion.split(":"):
        total = total * 60 + int(parte)
    return total


def es_nd(licencia):
    return "-nd" in (licencia or "").lower()


def metricas(gen, sp, grabaciones_chile):
    todas = consultar(f'gen:"{gen}" sp:"{sp}" grp:birds')
    durs = [segundos(g["length"]) for g in todas if g.get("length")]
    subida = lambda g: g.get("uploaded") or g.get("date") or ""

    sin_nd_previas = [
        g for g in todas if not es_nd(g.get("lic")) and subida(g) < CORTE_BIRDSET
    ]
    posteriores = [g for g in todas if subida(g) >= CORTE_BIRDSET]
    n_xcl = min(len(sin_nd_previas), MAX_XCL)
    min_xcl = sum(segundos(g["length"]) for g in sin_nd_previas if g.get("length")) / 60

    return {
        "especie": f"{gen} {sp}",
        "nombre_en": todas[0].get("en", "") if todas else "",
        "n_total_xc": len(todas),
        "n_en_chile": grabaciones_chile,
        "minutos_total": round(sum(durs) / 60, 1),
        "mediana_seg": round(statistics.median(durs), 1) if durs else 0,
        "n_calidad_AB": sum(1 for g in todas if g.get("q") in ("A", "B")),
        "n_grabadores": len({g.get("rec") for g in todas}),
        "n_xcl_aprox": n_xcl,
        "minutos_xcl_aprox": round(min_xcl, 1),
        "n_posteriores_corte": len(posteriores),
        "bajo_umbral": n_xcl < UMBRAL_GRABACIONES or min_xcl < UMBRAL_MINUTOS,
    }


def main():
    chile = consultar('cnt:"chile" grp:birds')
    conteo = {}
    for g in chile:
        clave = (g["gen"], g["sp"])
        if g["sp"].lower() in ("mystery", "sp", ""):
            continue
        conteo[clave] = conteo.get(clave, 0) + 1
    print(f"{len(conteo)} especies con grabaciones en Chile")

    filas = []
    for i, ((gen, sp), n_cl) in enumerate(sorted(conteo.items()), 1):
        print(f"[{i}/{len(conteo)}] {gen} {sp}")
        try:
            filas.append(metricas(gen, sp, n_cl))
        except requests.HTTPError as e:
            print(f"  error: {e}")
        time.sleep(1)

    filas.sort(key=lambda f: f["n_xcl_aprox"])
    with open("metricas_xc_chile.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(filas[0].keys()))
        w.writeheader()
        w.writerows(filas)
    print("Listo: metricas_xc_chile.csv")


if __name__ == "__main__":
    main()
```

# Imaginación y "dreaming" en modelos de IA: qué aportan al caso de las aves chilenas

Material de apoyo para estudiantes. Tercer documento de la serie sobre datos sintéticos para clasificar cantos de aves chilenas. Reúne trabajos recientes que usan las metáforas de "imaginación", "sueño" o "dreaming" en modelos de lenguaje y explica qué ideas se pueden trasladar al experimento con EfficientNet-B1-BirdSet-XCL.

Fecha: noviembre de 2026.

---

## 1. La idea común

Todos estos trabajos comparten un patrón: el modelo **genera ejemplos nuevos** a partir de lo que ya sabe, **selecciona** los que sirven y **se reentrena** con ellos, normalmente mezclados con datos reales. Las palabras "imaginación" o "sueño" son metáforas inspiradas en el aprendizaje humano, no mecanismos equivalentes al sueño biológico.

Ese patrón es el mismo que se quiere probar con las aves chilenas: generar cantos sintéticos para compensar la escasez de grabaciones reales. La diferencia es que estos trabajos operan sobre texto o imágenes y con modelos de lenguaje, por lo que sirven como **inspiración metodológica**, no como evidencia de que el enfoque funcione en bioacústica.

---

## 2. Los trabajos

### 2.1 Language Models Need Sleep (Behrouz, Hashemi y Mirrokni, 2026)

Plantea el aprendizaje continuo como un paradigma de "sueño" con dos etapas:

- **Consolidación de memoria (Knowledge Seeding):** las memorias de una versión más pequeña del modelo se destilan hacia una red más grande, que tiene más capacidad y preserva lo aprendido.
- **Dreaming:** una etapa separada que genera ejemplos sintéticos condicionados al contexto de la tarea y selecciona los que mejoran el modelo.

**Resultado clave:** en preentrenamiento continuo, la versión completa obtiene 46,2, frente a 43,2 del método de comparación (SEAL). Sin la etapa de dreaming, baja a 36,2. La generación y selección de sintéticos explica buena parte de la mejora.

**Límite:** la evidencia respalda el enfoque en las tareas estudiadas, pero no demuestra que resuelva el olvido catastrófico en general.

### 2.2 AlphaLLM: Toward Self-Improvement of LLMs via Imagination, Searching, and Criticizing (NeurIPS 2024)

Propone un ciclo de tres componentes inspirado en el self-play de AlphaGo:

- **Imaginación:** sintetiza prompts como nuevos ejemplos de aprendizaje, para enfrentar la escasez de datos anotados de calidad.
- **Búsqueda:** explora respuestas mejores con búsqueda en árbol (MCTS).
- **Críticos:** evalúan esas respuestas y guían la búsqueda.

Tras dos iteraciones con prompts sintéticos, los autores reportan un desempeño comparable al de GPT-4 en las tareas evaluadas.

### 2.3 Dreaming Out Loud (AlKhamissi et al., 2024)

Un modelo pequeño de visión y lenguaje genera sus propias descripciones para 1,1 millones de imágenes sin etiquetar, unos 42 millones de palabras sintéticas, y luego se reentrena con una mezcla de ese texto y los datos reales. Las ablaciones muestran que incluir los datos sintéticos mantuvo o mejoró el desempeño en todos los benchmarks de lenguaje evaluados.

Lo importante aquí es que **los datos de entrada son reales** (imágenes sin etiquetar); lo sintético son las etiquetas. Es una forma de autoentrenamiento o pseudoetiquetado.

### 2.4 Dreams de Anthropic (2026)

No es un paper sino una función anunciada el 6 de mayo de 2026 como vista previa para Claude Managed Agents. Consolida la memoria de un agente de forma offline: lee la memoria y las transcripciones pasadas, fusiona duplicados, reemplaza entradas obsoletas o contradictorias y propone ideas nuevas. No genera datos de entrenamiento, por lo que su relación con este caso es solo conceptual (ver sección 3.4).

---

## 3. Cómo se traslada al caso de las aves chilenas

### 3.1 Del "dreaming" a un ciclo de generar y seleccionar

La lección central de Language Models Need Sleep es que **generar no basta: hay que seleccionar**. Para las aves, eso se traduce en:

1. **Generar** cantos sintéticos de la especie objetivo (por ejemplo, el chuncho) con un generador como ECOGEN o BirdDiff, condicionado en la especie.
2. **Seleccionar** con criterios explícitos:
   - confianza de EfficientNet-B1-BirdSet-XCL para la especie objetivo;
   - distancia de los embeddings sintéticos a los reales, para descartar muestras fuera de distribución y también copias casi idénticas (memorización);
   - revisión de una muestra por una persona con experiencia en ornitología.
3. **Reentrenar** con la mezcla real más sintético seleccionado.

**Diseño experimental inspirado en la ablación del paper:** comparar cuatro condiciones:

- solo datos reales;
- reales con aumentación clásica;
- reales con sintéticos sin seleccionar;
- reales con sintéticos seleccionados.

Si la cuarta condición supera a la tercera, hay evidencia de que la selección importa, igual que en el paper.

### 3.2 De AlphaLLM: el crítico debe ser independiente

En AlphaLLM, los críticos guían qué ejemplos sirven. En el caso de las aves, el crítico natural es el propio clasificador, pero eso introduce un riesgo: **si el mismo modelo genera, filtra y aprende, puede reforzar sus propios errores**. Por ejemplo, podría conservar solo los sintéticos que ya reconoce bien, que son justamente los que menos aportan.

Mitigaciones:

- usar un crítico distinto del modelo que se entrena (otro modelo de BirdSet, como ConvNeXt, o un modelo externo como Perch o BirdNET);
- conservar también algunos sintéticos de confianza media, que suelen ser los más informativos;
- validar siempre con un set de prueba real e independiente.

### 3.3 De Dreaming Out Loud: pseudoetiquetar audio real

Esta es probablemente la idea más prometedora y la más barata. En vez de inventar audio, se aprovechan **grabaciones reales sin etiquetar**, por ejemplo paisajes sonoros de grabadoras autónomas en bosques o humedales chilenos:

1. EfficientNet-B1-BirdSet-XCL clasifica segmentos de 5 segundos de esas grabaciones.
2. Se conservan los segmentos con alta confianza para la especie objetivo.
3. Una persona verifica una muestra para estimar la tasa de error de las pseudoetiquetas.
4. Se reentrena con los datos reales etiquetados más los pseudoetiquetados.

Ventaja: el audio tiene el ruido, la acústica y los hábitats reales de Chile, que es justo lo que más le cuesta al modelo. Riesgo: si el modelo confunde especies parecidas (por ejemplo, churrines del género *Scytalopus*), propaga esos errores. Por eso la verificación humana y un umbral alto son esenciales.

Esta condición puede agregarse como una quinta rama del experimento: **reales con pseudoetiquetados**, comparada contra **reales con sintéticos**.

### 3.4 De la consolidación de memoria: curar el dataset

Tanto el Knowledge Seeding como la función Dreams de Anthropic apuntan a **ordenar lo aprendido**: fusionar duplicados, eliminar lo obsoleto o contradictorio y preservar lo valioso. En el caso de las aves, la analogía útil es la curaduría periódica del dataset:

- eliminar duplicados (la misma grabación subida dos veces o segmentos casi idénticos);
- revisar etiquetas contradictorias entre fuentes (Xeno-Canto, Macaulay Library, grabaciones propias);
- documentar qué datos se agregaron en cada iteración y con qué criterio.

La idea de destilar un modelo pequeño en uno grande es menos directa aquí y no se recomienda como punto de partida.

---

## 4. Advertencia general: el colapso por datos autogenerados

Entrenar repetidamente con datos generados por modelos puede degradar el desempeño: la distribución aprendida se estrecha y se pierden los casos poco frecuentes. Este fenómeno, conocido como colapso del modelo, está documentado para modelos generativos de texto e imagen (Shumailov et al., 2024).

Para este experimento implica tres reglas:

1. **Mantener siempre datos reales** en el entrenamiento y no reemplazarlos por sintéticos.
2. **Limitar las iteraciones** de generar, seleccionar y reentrenar, y medir en cada una.
3. **Evaluar solo con audio real**, separado por grabación y por grabador, nunca con sintéticos.

---

## 5. Resumen para el experimento

| Idea de la literatura | Traducción al caso de las aves | Rama experimental |
|---|---|---|
| Dreaming con selección (Sleep) | Generar cantos y filtrar por confianza, embeddings y revisión humana | Reales con sintéticos seleccionados vs. sin seleccionar |
| Críticos independientes (AlphaLLM) | Filtrar con un modelo distinto del que se entrena | Variante del filtro |
| Autosíntesis de etiquetas (Dreaming Out Loud) | Pseudoetiquetar grabaciones reales chilenas sin etiquetar | Reales con pseudoetiquetados |
| Consolidación de memoria (Sleep, Dreams) | Curar el dataset entre iteraciones | Práctica de trabajo, no rama |

---

## Referencias y enlaces

- Behrouz, A., Hashemi, F. y Mirrokni, V. (2026). Language Models Need Sleep: Learning to Self-Modify and Consolidate Memories. [arXiv:2606.03979](https://arxiv.org/abs/2606.03979)
- Toward Self-Improvement of LLMs via Imagination, Searching, and Criticizing (AlphaLLM). NeurIPS 2024. [PDF](https://proceedings.neurips.cc/paper_files/paper/2024/file/5e5853f35164e434015716a8c2a66543-Paper-Conference.pdf)
- AlKhamissi, B., Tang, Y., Gökce, A., Mehrer, J. y Schrimpf, M. (2024). Dreaming Out Loud: A Self-Synthesis Approach For Training Vision-Language Models With Developmentally Plausible Data. [Resumen en Lacuna](https://lacuna.tiptreesystems.com/work/dreaming-out-loud-a-self-synthesis-approach-for-training-vision-language-models/wrk_538d01cbb19036c7a9623fa7a485bcb3)
- Anthropic (2026). Dreams, función de consolidación de memoria para Claude Managed Agents (vista previa). Descrita en: ZenBrain: A Neuroscience-Inspired 7-Layer Memory Architecture for Autonomous AI Systems. [arXiv:2604.23878](https://arxiv.org/abs/2604.23878)
- Shumailov, I. et al. (2024). AI models collapse when trained on recursively generated data. *Nature*, 631, 755-759. [doi:10.1038/s41586-024-07566-y](https://doi.org/10.1038/s41586-024-07566-y)

### Documentos anteriores de la serie

- Datos sintéticos para clasificar cantos de aves chilenas (`datos-sinteticos-aves-chilenas.md`)
- Selección de especies chilenas para probar datos sintéticos (`seleccion-especies-aves-chilenas.md`)